# 01. Chạy baseline

Notebook này chạy hai cấu hình nền cho phần deep learning: `frozen` và `full_finetune`. Hai baseline này tạo mốc so sánh để đánh giá LoRA có giữ được hiệu năng tốt trong khi giảm số tham số cần huấn luyện hay không.

Ý nghĩa của hai cấu hình:

- `frozen`: đóng băng encoder, chỉ huấn luyện classifier. Đây là baseline rất nhẹ.
- `full_finetune`: cập nhật toàn bộ model. Đây là baseline mạnh nhưng tốn nhiều tham số trainable nhất.

Sau khi chạy, mỗi thí nghiệm sẽ sinh thư mục `results/runs/<run_name>/` chứa `metrics.json`, `config.resolved.json` và checkpoint tốt nhất.

## Bước 1. Đặt đúng thư mục làm việc

Cell này bảo đảm notebook luôn chạy từ thư mục `deep-learning`, kể cả khi người dùng mở notebook từ thư mục `notebooks`. Nhờ vậy các đường dẫn như `configs/frozen.yaml` và `results/runs` sẽ luôn đúng.

In [1]:
from pathlib import Path
import os
import sys

print(f"Python đang dùng: {sys.executable}")
if ".venv" not in sys.executable:
    raise RuntimeError(
        "Notebook đang dùng sai kernel. Hãy chọn kernel 'Lab 3 Deep Learning' "
        "hoặc interpreter deep-learning/.venv/Scripts/python.exe, rồi Restart kernel."
    )

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
print(f"Thư mục làm việc: {Path.cwd()}")

Python đang dùng: d:\HK6\Machine learning\Lab_3\csc14005-introduction-to-machine-learning\lab-3\deep-learning\.venv\Scripts\python.exe
Thư mục làm việc: d:\HK6\Machine learning\Lab_3\csc14005-introduction-to-machine-learning\lab-3\deep-learning


## Bước 2. Chạy hai baseline

Cell này đọc từng file config, khởi tạo model/dataset theo config rồi gọi `run_training`. Với mỗi cấu hình, notebook in ra:

- `run_name`: tên thí nghiệm.
- `best_acc`: validation accuracy tốt nhất trong quá trình train.
- `trainable_params`: số tham số được cập nhật.

Trong lần chạy đã dùng cho báo cáo, frozen đạt khoảng `0.6433` accuracy với `258` tham số trainable, còn full fine-tune đạt khoảng `0.7661` accuracy với `4,386,178` tham số trainable.

In [2]:
from src.config import load_config
from src.train import run_training

baseline_configs = [
    "configs/frozen.yaml",
    "configs/full_finetune.yaml",
]

baseline_metrics = []
for config_path in baseline_configs:
    print(f"\nĐang chạy {config_path}")
    metrics = run_training(load_config(config_path))
    baseline_metrics.append(metrics)
    print(
        metrics["run_name"],
        "best_acc=", round(metrics["best_val_accuracy"], 4),
        "trainable_params=", metrics["model"]["trainable_params"],
    )

d:\HK6\Machine learning\Lab_3\csc14005-introduction-to-machine-learning\lab-3\deep-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Đang chạy configs/frozen.yaml


d:\HK6\Machine learning\Lab_3\csc14005-introduction-to-machine-learning\lab-3\deep-learning\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LAPTOP HP\.cache\huggingface\hub\models--prajjwal1--bert-tiny. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the '

frozen best_acc= 0.6525 trainable_params= 258

Đang chạy configs/full_finetune.yaml


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


full_finetune best_acc= 0.7569 trainable_params= 4386178


## Kết quả cần kiểm tra

Sau cell trên, kiểm tra các file sau:

- `results/runs/frozen/metrics.json`
- `results/runs/full_finetune/metrics.json`

Nếu hai file này tồn tại và có `best_val_accuracy`, pipeline baseline đã chạy thành công. Các số này sẽ được dùng làm mốc để so sánh với LoRA trong notebook tiếp theo.